### Car Failure Prediction: EDA, Data Preprocessing, and Classification Target Definition

#### The objective of this project is to help identify vehicles that are likely to fail based on vehicle characteristics, operating conditions, and usage-related features. By predicting whether a car will fail or pass, the business can prioritise preventive maintenance, reduce unexpected breakdowns, and improve operational reliability. 

#### What you will find in this notebook:
- Load and inspect the dataset from Failure.csv
- Clean column names and identify feature groups
- Assess missing values and invalid readings
- Handle data preprocessing decisions for temperature, RPM, fuel consumption, and membership
- Explore numerical and categorical feature distributions
- Define a binary Fail/Pass classification target
- Check class balance between failed and passed cars
- Prepare the dataset for machine learning classification




In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 as sq3
import duckdb

#### Load data from SQLite

The dataset is retrieved from the `failure` table inside `Failure.csv`

In [3]:
import duckdb

con = duckdb.connect("failure.db")

# 2. Import the CSV file into a real DuckDB table
con.execute("""
CREATE OR REPLACE TABLE failure AS
SELECT *
FROM read_csv_auto('Failure.csv')
""")


#### Load the dataset

In [4]:
query = """
select *
from failure

""" 

con.execute(query).df()

,Car ID,Model,Color,Temperature,RPM,Factory,Usage,Fuel consumption,Membership,Failure A,Failure B,Failure C,Failure D,Failure E
0,CAR-00001,Pickup,Yellow,92.0,4846.0,Factory C,High,9.41,None,0,1,0,0,0
1,CAR-00002,Sedan,Yellow,100.4,5057.0,Factory D,Very High,9.54,Basic,0,0,0,0,0
2,CAR-00003,Pickup,Green,64.9,2770.0,Factory C,Low,NaN,None,0,0,0,0,0
3,CAR-00004,Hatchback,Gray,76.0,2749.0,Factory D,Medium,8.07,Gold,0,1,0,0,0
4,CAR-00005,SUV,Blue,78.2,4755.0,Factory A,Medium,9.45,None,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10076,CAR-10077,Wagon,Blue,78.4,2697.0,Factory C,Low,6.91,Platinum,0,0,0,0,0
10077,CAR-10078,Van,Red,79.2,2554.0,Factory B,Low,7.64,Silver,0,0,0,0,0
10078,CAR-10079,Hatchback,Blue,77.0,4292.0,Factory A,High,8.45,Gold,0,0,0,0,0
10079,CAR-10080,Hatchback,Red,92.9,4349.0,Factory A,High,9.94,Gold,0,0,0,1,0


#### Data Quality Check

In [5]:
query = """
select sum(case when "Temperature" is null or "Temperature" <= 0 then 1 else 0 end) as bad_temp,
sum(case when "RPM" is null or "RPM" <= 0 then 1 else 0 end) as bad_rpm,
sum(case when "Fuel consumption" is null or "Fuel consumption" <= 0 then 1 else 0 end) as bad_fuel_consumption,
sum(case when "Membership" is null then 1 else 0 end) as bad_membership,
sum(case when "color" is null then 1 else 0 end) as bad_color,
sum(case when "Factory" is null then 1 else 0 end) as bad_factory,
sum(case when "Usage" is null then 1 else 0 end) as bad_usage,
sum(case when "Model" is null then 1 else 0 end) as bad_model
from failure

""" 

con.execute(query).df()

,bad_temp,bad_rpm,bad_fuel_consumption,bad_membership,bad_color,bad_factory,bad_usage,bad_model
0,213.0,146.0,236.0,2032.0,0.0,0.0,0.0,0.0


#### Change columns name from uppercase to lowercase for easier handling

In [6]:
con.execute('alter table failure rename column "Car ID" to car_id')
con.execute('alter table failure rename column "Model" to model')
con.execute('alter table failure rename column "Color" to color')
con.execute('alter table failure rename column "RPM" to rpm')
con.execute('alter table failure rename column "Factory" to factory')
con.execute('alter table failure rename column "Usage" to usage')
con.execute('alter table failure rename column "Fuel consumption" to fuel_consumption')
con.execute('alter table failure rename column "Membership" to membership')
con.execute('alter table failure rename column "Failure A" to failure_a')
con.execute('alter table failure rename column "Failure B" to failure_b')
con.execute('alter table failure rename column "Failure C" to failure_c')
con.execute('alter table failure rename column "Failure D" to failure_d')
con.execute('alter table failure rename column "Failure E" to failure_e')
con.execute('alter table failure rename column "Temperature" to temp')

#### Impute median into null values to prevent data right skewed

In [7]:
con.execute("""
update failure as f 
set rpm = m.median_rpm 
from (select model, median(rpm) as median_rpm 
from failure 
where "rpm" > 0 
group by model) as m 
where f.model = m.model and (f.rpm <= 0 or f.rpm is null)
""")

con.execute(""" 
update failure as f 
set fuel_consumption = median_fuel 
from (select model, median(fuel_consumption) as median_fuel 
from failure 
where "fuel_consumption" > 0 
group by model) as fc 
where f.model = fc.model and (f.fuel_consumption <= 0 or f.fuel_consumption is null) 
""")

con.execute("""
update failure as f
set temp = median_temp
from (select model, median(temp) as median_temp
from failure
where "temp" > 0
group by model) as m
where f.model = m.model and (f.temp is null or f.temp <= 0)   
""")



#### Observation: Missingness concentrated in `Membership`

The missing-value pattern suggests that missing entries are limited to a single feature (`Membership`).
In this notebook, missing membership is treated as an explicit category (`No Membership`), which is often a practical approach for categorical variables when the absence itself can carry information.

#### Missing Values Overview

Before any modeling or transformation, we inspect missing values to understand which features require cleaning or imputation. 
The heatmap below highlights the positions of missing entries. 

In [8]:
con.execute("""
update failure
set membership = 'No Membership'
where membership is null or membership = 'None'
""")

#### Change membership from none to No membership

In [9]:
query = """
select membership, count(*) as no_membership
from failure
group by membership

""" 

con.execute(query).df()

,membership,no_membership
0,Basic,2075
1,Gold,2017
2,Platinum,1990
3,Silver,1967
4,No Membership,2032


#### Data Quality check before feature engineering

In [10]:
query = """
select sum(case when "car_id" is null then 1 else 0 end) as car_id_null, 
sum(case when "model" is null then  1 else 0 end) as model_null,
sum(case when "color" is null then 1 else 0 end) as color_null,
sum(case when "temp" is null then 1 else 0 end) as temperature_null,
sum(case when "rpm" is null then 1  else 0 end) as rpm_null,
sum(case when "factory" is null then 1 else 0 end) as factory_null,
sum(case when "usage" is null then 1 else 0 end) as usage_null,
sum(case when "fuel_consumption" is null then 1 else 0 end) as fuel_consumption_null,
sum(case when "membership" is null then 1 else 0 end) as membership_null,
sum(case when "failure_a" is null then 1 else 0 end) as failure_a_null,
sum(case when "failure_b" is null then 1 else 0 end) as failure_b_null,
sum(case when "failure_c" is null then 1 else 0 end) as failure_c_null,
sum(case when "failure_d" is null then 1 else 0 end) as failure_d_null,
sum(case when "failure_e" is null then 1 else 0 end) as failure_e_null
from failure
"""

con.execute(query).df()

,car_id_null,model_null,color_null,temperature_null,rpm_null,factory_null,usage_null,fuel_consumption_null,membership_null,failure_a_null,failure_b_null,failure_c_null,failure_d_null,failure_e_null
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Define feature groups

We separate:

- **Targets**: the failure indicator columns (`Failure A` ... `Failure E`)
- **Features**: all remaining columns

We also identify numerical vs categorical features for EDA

#### Target Definition

The original dataset contains five failure indicators: failure_a, failure_b, failure_c, failure_d, and failure_e.

For classification modeling, these are combined into one binary target

- Fail: if any failure column equals to 1
- Pass: if all failure columns equal to 0

In [11]:
query = """
select case when coalesce(failure_a, 0) + coalesce(failure_b, 0) + coalesce(failure_c, 0) + coalesce(failure_d, 0) + coalesce(failure_e, 0) > 0 then 'fail' else 'pass' end as car_status, count(*) as count, 
round(100.0 * count(*) / sum(count(*)) over(), 2) as pct
from failure
group by car_status

"""

con.execute(query).df()

,car_status,count,pct
0,fail,3614,35.85
1,pass,6467,64.15


#### SUV has the highest failure rate of 38.24%, meaning SUV are the most failure-prone model group in this daaset. Hatchback has the lowest failure rate at 32.49%. This suggest model type may be useful feature for predicting whether a car will fail or pass.

In [12]:
query = """
with total_failure_rate as (
select model as models, count(*) as total_cars
from failure
group by model),
failure_rate as (
select model as failed_models, count(*) as failed_cars
from failure 
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by model)
select *, round(100.0 * failed_cars / total_cars, 2) as failure_rate_pct
from failure_rate as fr
left join total_failure_rate as tfr on fr.failed_models = tfr.models
order by failure_rate_pct desc


"""

con.execute(query).df()

,failed_models,failed_cars,models,total_cars,failure_rate_pct
0,SUV,574,SUV,1501,38.24
1,Coupe,508,Coupe,1355,37.49
2,Pickup,502,Pickup,1366,36.75
3,Van,555,Van,1513,36.68
4,Wagon,518,Wagon,1431,36.20
5,Sedan,481,Sedan,1450,33.17
6,Hatchback,476,Hatchback,1465,32.49


#### Factory B has the highest failure rate in producing vehicles while vehicle A has the lowest failure rate in producing vehicles. This suggests that factory source may be associated with failure risk and could be useful as a predictive feature for the fail/pass classification model

In [13]:
query = """
with failure_rate_factory as (select factory as failed_factory, count(*) as no_failed_factory
from failure
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by factory),
total_factory as (select factory, count(*) as total_factory
from failure
group by factory)
select *, round(100.0 * no_failed_factory / total_factory, 2) as failed_factory_pct
from failure_rate_factory as frf 
left join total_factory as tf on frf.failed_factory = tf.factory
order by failed_factory_pct desc
"""

con.execute(query).df()

,failed_factory,no_failed_factory,factory,total_factory,failed_factory_pct
0,Factory B,739,Factory B,1937,38.15
1,Factory D,772,Factory D,2083,37.06
2,Factory E,723,Factory E,2067,34.98
3,Factory C,694,Factory C,1999,34.72
4,Factory A,686,Factory A,1995,34.39


#### The number of usage which is very high lead to a very high failure rate as compared to low usage which only has 17.98% of the failure rate

In [14]:
query = """
with total_usage as (select usage, count(*) as total_no_usage
from failure
group by usage),
usage_group as (select usage as failed_usage, count(*) as no_failed_usage
from failure
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by usage 
)
select *, round(100.0 * no_failed_usage / total_no_usage, 2) as failed_usage_pct
from usage_group as ug
left join total_usage as tu on ug.failed_usage = tu.usage
order by failed_usage_pct desc


"""

con.execute(query).df()

,failed_usage,no_failed_usage,usage,total_no_usage,failed_usage_pct
0,Very High,700,Very High,981,71.36
1,High,1213,High,2503,48.46
2,Medium,1236,Medium,4011,30.82
3,Low,465,Low,2586,17.98


#### Total number of model pass or fail status

In [15]:
query = """
select model, case when failure_a + failure_b + failure_c + failure_d + failure_e > 0 then 'fail' else 'pass' end as car_status, count(*) as count, round(100.0 * count(*) / sum(count(*)) over(), 2) as pct
from failure
group by model, car_status
order by count desc
"""

con.execute(query).df()

,model,car_status,count,pct
0,Hatchback,pass,989,9.81
1,Sedan,pass,969,9.61
2,Van,pass,958,9.50
3,SUV,pass,927,9.20
4,Wagon,pass,913,9.06
5,Pickup,pass,864,8.57
6,Coupe,pass,847,8.40
7,SUV,fail,574,5.69
8,Van,fail,555,5.51
9,Wagon,fail,518,5.14


#### Feature engineering classifying rpm, temperature and fuel consumption in terms of low, medium and high. RPM, temperature and fuel consumptions were converted into risk groups to make continuous values easier to interpret for failure-risk analysis. These grouped features may help to identify whether extreme operating conditions are associated with higher failure rates.

In [16]:
query = """
select *, case when rpm between 0 and 1500 then 'Low'
when rpm between 1501 and 3000 then 'Medium'
when rpm between 3001 and 4500 then 'High'
when rpm > 4500 then 'Danger' end as rpm_class,

case when temp < 75 then 'Low'
when temp >= 75 and temp < 105 then 'Medium'
when temp >= 105 then 'High' end as temp_class,

case when fuel_consumption < 5 then 'Low'
when fuel_consumption >= 5 and fuel_consumption <= 8 then 'Medium' 
when fuel_consumption > 8 then 'High' end as fuel_class
from failure
 
"""

failure_fe = con.execute(query).df()
con.register('failure_fe', failure_fe)

#### RPM class has categorised as danger zone with high RPM has the highest of failure rate

In [17]:
query = """
with total_rpm as (select rpm_class as total_rpm_class, count(*) as total_no_rpm
from failure_fe
group by rpm_class),
failed_rpm as (select rpm_class, count(*) as no_failed_rpm
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by rpm_class)
select *, round(100.0 * no_failed_rpm / total_no_rpm, 2) as failed_rpm_pct
from failed_rpm as fr
left join total_rpm as tr on fr.rpm_class = tr.total_rpm_class 
order by failed_rpm_pct desc
"""

con.execute(query).df()

,rpm_class,no_failed_rpm,total_rpm_class,total_no_rpm,failed_rpm_pct
0,Danger,628,Danger,973,64.54
1,High,2011,High,4941,40.70
2,Medium,933,Medium,3880,24.05
3,Low,42,Low,287,14.63


#### Customers who subscribed for platinum membership has the highest car failure rate as compared to basic membership members

In [24]:
query = """
with all_membership as (select membership, count(*) as no_total_membership
from failure_fe
group by membership),
failed_membership as (select membership as fail_membership, count(*) as no_failed_membership
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1
group by fail_membership)
select *, round(100.0 * no_failed_membership / no_total_membership, 2) as membership_pct
from failed_membership as fm
left join all_membership as am on fm.fail_membership = am.membership
order by membership_pct desc

"""

con.execute(query).df()

,fail_membership,no_failed_membership,membership,no_total_membership,membership_pct
0,Platinum,721,Platinum,1990,36.23
1,No Membership,734,No Membership,2032,36.12
2,Silver,709,Silver,1967,36.04
3,Gold,716,Gold,2017,35.50
4,Basic,734,Basic,2075,35.37


#### Cars in the high fuel consumption group have the highest failure rate at 42.28%. This suggests that high fuel consumption may be associated with increased failure risk, although further analysis is needed to confirm causality. 

In [30]:
query = """
with total_fuel as (select fuel_class, count(*) as no_total_fuel
from failure_fe
group by fuel_class),
failed_fuel as (select fuel_class as failed_fuel_class, count(*) as failed_fuel
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_e = 1 or failure_d = 1 or failure_e = 1
group by fuel_class)
select *, round(100.0 * failed_fuel / no_total_fuel, 2) as fuel_pct
from failed_fuel as ff
left join total_fuel as tf on ff.failed_fuel_class = tf.fuel_class 
order by fuel_pct desc

"""

con.execute(query).df()

,failed_fuel_class,failed_fuel,fuel_class,no_total_fuel,fuel_pct
0,High,2986,High,7063,42.28
1,Medium,615,Medium,2926,21.02
2,Low,13,Low,92,14.13


#### The high temperature group has the highest failure rate at 67%, compared with 44% for medium temperature and 23% for low temperature. This suggests that temperature group may be a useful predictor for the Fail/Pass classification task. Although the sample size is small with only 64 cars on the highest failure rate, this result should be interpreted carefully because small group sizes can make percentages unstable.

In [35]:
query = """
with total_temp as (select temp_class as total_temp_class, count(*) as total_no_temp
from failure_fe
group by total_temp_class),
failed_temp as (select temp_class, count(*) as no_failed_temp
from failure_fe
where failure_a = 1 or failure_b = 1 or failure_c = 1 or failure_d = 1 or failure_e = 1 
group by temp_class)
select *, round(100.0 * no_failed_temp / total_no_temp) as temp_pct
from failed_temp as ft
left join total_temp as tt on ft.temp_class = tt.total_temp_class
order by temp_pct desc

"""

con.execute(query).df()

,temp_class,no_failed_temp,total_temp_class,total_no_temp,temp_pct
0,High,43,High,64,67.0
1,Medium,2602,Medium,5876,44.0
2,Low,969,Low,4141,23.0


#### Failure D is the most common failure type, with 1094 occurrences. Failure C is the least common failure type, with 809 occurrences. This suggests that Failure D may require the highest maintenance priority.

In [48]:
query = """
select sum(case when failure_a = 1 then 1 else 0 end) as no_failure_a, sum(case when failure_b = 1 then 1 else 0 end) as no_failure_b, 
sum(case when failure_c = 1 then 1 else 0 end) as no_failure_c, sum(case when failure_d = 1 then 1 else 0 end) as no_failure_d, sum(case when failure_e = 1 then 1 else 0 end) as no_failure_e,
from failure_fe
"""

con.execute(query).df()

,no_failure_a,no_failure_b,no_failure_c,no_failure_d,no_failure_e
0,861.0,1044.0,809.0,1094.0,835.0


#### This analysis check whether failed vehicles usually experience one failure type or multiple failure types. Single failures suggest isolated issues, while multiple failures may indicate broader mechanical problems or related failure patterns

In [71]:
query = """
select case when failure_a + failure_b + failure_c + failure_d + failure_e = 0 then 'No Failure'
when failure_a + failure_b + failure_c + failure_d + failure_e = 1 then 'Single Failure' else 'Multiple Failures' end as Failure_group, count(*) as no_failure_group, round(100.0 * count(*) / sum(count (*)) over(), 2) as pct
from failure_fe
group by failure_group
order by pct desc
"""

con.execute(query).df()

,Failure_group,no_failure_group,pct
0,No Failure,6467,64.15
1,Single Failure,2738,27.16
2,Multiple Failures,876,8.69


#### Key Findings

- The dataset is balanced because the fail class is 35.85% while the pass class is around 64.15%
- The high risk car model is SUV targeted at 38.24%
- High risk factory B manufacture about 38.15% failed cars
- Very high usage group caused 71.36% of the car failure
- High fuel consumption group has 42.28% under the car failure category